## Concept 5 — Token Buffer with trim_messages

### What is it?
Same `trim_messages` function as Concept 2, but now counting **real tokens** instead of message count.
Keeps as many recent messages as fit within a token budget.

### Key Parameters
```python
trim_messages(
    messages,
    max_tokens=300,
    token_counter='approximate',   # fast estimation: ~4 chars = 1 token
    strategy='last',               # keep most recent
    start_on='human',              # always begin with a HumanMessage
    include_system=True,           # keep SystemMessage even if old
)
```

### Token vs Message Count
```
Window Memory (count messages):   k=3 keeps 3 turns regardless of their size
Token Buffer (count tokens):      max=300 keeps whatever fits — predictable cost
```

### Limitation (what Concept 6 fixes)
Still drops oldest messages by recency — cannot retrieve old but relevant turns.
→ Concept 6 uses vector similarity to fetch the most semantically relevant past turns.

In [ ]:
print('hi')

In [ ]:
import sys, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, '.')

from MemoryUtils import get_llm, TEST_TURNS, make_history, load_test_history, run_recall_test
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import trim_messages, HumanMessage, AIMessage

llm = get_llm()
print('LLM ready')

### Step 1 — trim_messages with Token Counting

In [ ]:
history = make_history()
load_test_history(history)

MAX_TOKENS = 150

trimmed = trim_messages(
    history.messages,
    max_tokens=MAX_TOKENS,
    token_counter='approximate',
    strategy='last',
    start_on='human',
)

all_tokens  = sum(len(m.content) for m in history.messages) // 4
kept_tokens = sum(len(m.content) for m in trimmed) // 4

print(f'All messages   : {len(history.messages)} messages  ~{all_tokens} tokens')
print(f'After trim     : {len(trimmed)} messages  ~{kept_tokens} tokens  (budget={MAX_TOKENS})')
print()
print('Kept messages:')
for msg in trimmed:
    role = 'Human' if isinstance(msg, HumanMessage) else 'AI'
    print(f'  [{role}] {msg.content[:70]}...' if len(msg.content) > 70 else f'  [{role}] {msg.content}')

### Step 2 — Effect of Different Token Budgets

In [ ]:
print('After loading 5 turns, what each token budget retains:')
print('-' * 60)
for budget in [50, 100, 200, 500]:
    trimmed = trim_messages(
        history.messages,
        max_tokens=budget,
        token_counter='approximate',
        strategy='last',
        start_on='human',
    )
    kept_tok = sum(len(m.content) for m in trimmed) // 4
    oldest = trimmed[0].content[:45] + '...' if trimmed else 'empty'
    print(f'budget={budget:3d}: {len(trimmed):2d} msgs ({len(trimmed)//2} turns) ~{kept_tok:3d} tokens | oldest: "{oldest}"')

### Step 3 — Token vs Message Count: Why Tokens Win
One very long turn can blow the budget with message counting, but not with token counting.

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage

# Simulate a conversation with one very long AI response
lopsided = [
    HumanMessage(content='Hi'),
    AIMessage(content='Hello!'),
    HumanMessage(content='Explain Python in detail.'),
    AIMessage(content='Python is a high-level interpreted language ' * 20),  # very long!
    HumanMessage(content='Thanks.'),
    AIMessage(content='You are welcome!'),
]

# Message-count window (k=2 turns = 4 messages): includes the huge turn
msg_window = lopsided[-4:]
msg_tokens = sum(len(m.content) for m in msg_window) // 4

# Token trim (budget=100): drops the huge turn, keeps only what fits
tok_window = trim_messages(
    lopsided,
    max_tokens=100,
    token_counter='approximate',
    strategy='last',
    start_on='human',
)
tok_tokens = sum(len(m.content) for m in tok_window) // 4

print('Message window (k=2):  ', len(msg_window), 'messages  ~', msg_tokens, 'tokens  <- HUGE!')
print('Token window (100):    ', len(tok_window), 'messages  ~', tok_tokens, 'tokens  <- bounded')

### Full Function — chat_with_token_buffer

In [ ]:
MAX_TOKENS = 300

store = {}
def get_session_history(session_id: str) -> ChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

def trim_to_tokens(messages):
    return trim_messages(
        messages,
        max_tokens=MAX_TOKENS,
        token_counter='approximate',
        strategy='last',
        start_on='human',
    )

from langchain_core.runnables import RunnableLambda

prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a helpful assistant. Answer using the conversation history.'),
    MessagesPlaceholder(variable_name='history'),
    ('human', '{input}'),
])

chain = (
    {'input': lambda x: x['input'], 'history': lambda x: trim_to_tokens(x['history'])}
    | prompt
    | llm
)

chain_with_tokens = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key='input',
    history_messages_key='history',
)

SESSION = 'token_test'
for human, ai in TEST_TURNS:
    get_session_history(SESSION).add_user_message(human)
    get_session_history(SESSION).add_ai_message(ai)

def chat_with_token_buffer(query):
    result = chain_with_tokens.invoke(
        {'input': query},
        {'configurable': {'session_id': SESSION}}
    )
    return result.content

### Test — Recall with Token Budget (max=300)

In [ ]:
run_recall_test(chat_with_token_buffer, label='Concept 5 -- Token Buffer (trim_messages)')